# Bundle Migration: Export & Transform

## Overview

Export and transform dashboards using centralized configuration and helper modules.

**What this notebook does:**
1. Loads config from `config/config.yaml`
2. Discovers dashboards from source workspace
3. Exports dashboard JSONs and permissions
4. Applies catalog/schema transformations
5. Saves transformed files for bundle generation

**Next step:** Run `Bundle_02_Generate_and_Deploy.ipynb`

---

## Cell 0: Install Dependencies

In [ ]:
%pip install -U databricks-sdk pandas pyyaml "numpy<2" --quiet
dbutils.library.restartPython()

## Cell 1: Import Helpers & Setup

In [ ]:
# VERIFY APPROVED INVENTORY EXISTS AND IS RECENT

import datetime

print("="*70)
print("STEP 2: EXPORT & TRANSFORM")
print("="*70)
print("\n🔍 Verifying approved inventory...\n")

# Get configuration
volume_base = dbutils.widgets.get("volume_base")
inventory_path = dbutils.widgets.get("inventory_path")  # Should be dashboard_inventory_approved
approved_csv_path = f"{volume_base}/{inventory_path}/inventory_approved.csv"

# Check if file exists
try:
    file_info = dbutils.fs.ls(f"{volume_base}/{inventory_path}/")
    csv_found = False
    
    for file in file_info:
        if file.name == 'inventory_approved.csv':
            csv_found = True
            modified_time = datetime.datetime.fromtimestamp(file.modificationTime / 1000)
            age_hours = (datetime.datetime.now() - modified_time).total_seconds() / 3600
            
            print(f"✅ Approved inventory found")
            print(f"   Location: {approved_csv_path}")
            print(f"   Modified: {modified_time.strftime('%Y-%m-%d %H:%M:%S')}")
            print(f"   Age: {age_hours:.1f} hours ago")
            
            # Warn if file is old (older than 7 days)
            if age_hours > 168:
                print(f"\n⚠️  WARNING: File is {age_hours/24:.1f} days old")
                print(f"   Consider regenerating inventory if dashboards changed")
            
            # Load and show count
            df_approved = spark.read.csv(approved_csv_path, header=True, inferSchema=True)
            print(f"   Dashboards: {df_approved.count()}\n")
            break
    
    if not csv_found:
        raise FileNotFoundError("inventory_approved.csv not found")
        
except Exception as e:
    print("="*70)
    print("❌ ERROR: APPROVED INVENTORY NOT FOUND")
    print("="*70)
    print(f"\nExpected location: {approved_csv_path}")
    print("\n🎯 REQUIRED ACTIONS:")
    print("   1. Run Step 1: databricks bundle run inventory_generation -t dev")
    print("   2. Run Step 2: Open Bundle/Bundle_02_Review_and_Approve_Inventory.ipynb")
    print("              in Databricks UI and complete the review process")
    print(f"\nError details: {e}")
    print("="*70)
    raise FileNotFoundError(f"Approved inventory not found at {approved_csv_path}")

print("="*70)
print("✅ VERIFICATION PASSED - STARTING EXPORT")
print("="*70 + "\n")

In [ ]:
import sys
import os
import json
from pathlib import Path
import pandas as pd

# Dynamically locate helpers directory
print("=== HELPERS PATH RESOLUTION ===")

try:
    # In Databricks workspace/job context
    notebook_path = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
    print(f"Notebook path: {notebook_path}")
    
    # Add /Workspace prefix if not present
    if not notebook_path.startswith('/Workspace'):
        notebook_path = '/Workspace' + notebook_path
        print(f"Added /Workspace prefix: {notebook_path}")
    
    notebook_dir = os.path.dirname(notebook_path)
    bundle_parent = os.path.dirname(notebook_dir)
    print(f"Bundle parent: {bundle_parent}")
    
    # Add BUNDLE PARENT to sys.path (not the helpers dir itself)
    sys_path_entry = bundle_parent
    
except Exception as e:
    print(f"Workspace context error: {e}")
    # Fallback for local execution
    sys_path_entry = os.path.abspath(os.path.join(os.getcwd(), '..'))
    print(f"Using local fallback: {sys_path_entry}")

sys_path_entry = os.path.normpath(sys_path_entry)

# Add bundle parent to sys.path
print(f"Adding to sys.path: {sys_path_entry}")
if sys_path_entry not in sys.path:
    sys.path.insert(0, sys_path_entry)
    print("  Added to sys.path")

print(f"sys.path first 3 entries: {sys.path[:3]}")
print("=== END PATH RESOLUTION ===\n")

from helpers import (
    set_config,
    get_path,
    create_workspace_client,
    discover_dashboards,
    export_dashboard,
    get_dashboard_permissions,
    get_dashboard_schedules,
    load_mapping_csv,
    transform_dashboard_json,
    write_volume_file,
    ensure_directory_exists
)

print("✅ Helper modules imported")

## Cell 2: Load Configuration & Archive

## Cell 2.5: Configuration Validation

**Pre-flight checks before export & transform:**
- ✅ Workspace connectivity
- ✅ Volume paths accessible  
- ✅ Mapping CSV valid (if transformation enabled)
- ✅ Required permissions present

In [ ]:
# ============================================================================
# CONFIGURATION FROM PARAMETERS (databricks.yml)
# ============================================================================
# This notebook reads configuration from job parameters (widgets)
# All configuration is passed from databricks.yml

print("="*80)
print("LOADING CONFIGURATION FROM PARAMETERS")
print("="*80)

# Read all parameters from widgets
catalog = dbutils.widgets.get('catalog')
volume_base = dbutils.widgets.get('volume_base')
source_workspace_url = dbutils.widgets.get('source_workspace_url')
target_workspace_url = dbutils.widgets.get('target_workspace_url')
inventory_path_rel = dbutils.widgets.get('inventory_path')
exported_path_rel = dbutils.widgets.get('exported_path')
transformed_path_rel = dbutils.widgets.get('transformed_path')
mappings_path_rel = dbutils.widgets.get('mappings_path')
transformation_enabled = dbutils.widgets.get('transformation_enabled').lower() == 'true'
mapping_csv_path = dbutils.widgets.get('mapping_csv_path')
capture_permissions = dbutils.widgets.get('capture_permissions').lower() == 'true'
capture_schedules = dbutils.widgets.get('capture_schedules').lower() == 'true'

# Archive options (with defaults if not provided)
try:
    archive_old_files = dbutils.widgets.get('archive_old_files').lower() == 'true'
except:
    archive_old_files = True  # Default: enabled

try:
    archive_min_age_minutes = int(dbutils.widgets.get('archive_min_age_minutes'))
except:
    archive_min_age_minutes = 5  # Default: 5 minutes

try:
    force_archive_all = dbutils.widgets.get('force_archive_all').lower() == 'true'
except:
    force_archive_all = False  # Default: disabled

# Override age threshold if force_archive_all is enabled
if force_archive_all:
    archive_min_age_minutes = 0  # Archive everything

# Build full paths
inventory_path = f"{volume_base}/{inventory_path_rel}"
exported_path = f"{volume_base}/{exported_path_rel}"
transformed_path = f"{volume_base}/{transformed_path_rel}"
mappings_path = f"{volume_base}/{mappings_path_rel}"

# Build config dict for helper functions
config = {
    'source': {
        'workspace_url': source_workspace_url,
        'auth': {'method': 'oauth'}
    },
    'target': {
        'workspace_url': target_workspace_url,
        'auth': {'method': 'oauth'}
    },
    'paths': {
        'volume_base': volume_base,
        'inventory': inventory_path_rel,
        'exported': exported_path_rel,
        'transformed': transformed_path_rel,
        'mappings': mappings_path_rel
    },
    'dashboard_selection': {
        'method': 'inventory_csv',  # This notebook reads from approved inventory
        'catalog_filter': {
            'catalog': catalog
        }
    },
    'transformation': {
        'enabled': transformation_enabled,
        'mapping_csv': mapping_csv_path
    },
    'permissions': {
        'capture': capture_permissions
    },
    'schedules': {
        'capture': capture_schedules
    }
}

# Cache config for helper functions
set_config(config)

print("\n✅ Configuration loaded from parameters\n")
print(f"   Source: {source_workspace_url}")
print(f"   Target: {target_workspace_url}")
print(f"   Volume base: {volume_base}")
print(f"   Inventory: {inventory_path}")
print(f"   Transformation: {'Enabled' if transformation_enabled else 'Disabled'}")
print(f"   Capture permissions: {'Yes' if capture_permissions else 'No'}")
print(f"   Capture schedules: {'Yes' if capture_schedules else 'No'}")

# Ensure directories exist
print("\n📁 Ensuring directories exist...")
ensure_directory_exists(exported_path)
ensure_directory_exists(transformed_path)
print(f"   ✅ Export: {exported_path}")
print(f"   ✅ Transformed: {transformed_path}")

# DEBUG: Show archive configuration
print("\n🔍 DEBUG - Archive Configuration:")
print(f"   archive_old_files: {archive_old_files} (type: {type(archive_old_files)})")
print(f"   force_archive_all: {force_archive_all}")
print(f"   archive_min_age_minutes: {archive_min_age_minutes} {'(OVERRIDDEN TO 0 - FORCE ARCHIVE ALL)' if force_archive_all else ''}")

# ============================================================================
# Archive configuration loaded - archiving happens in transform step (before writing files)

In [ ]:
# Minimal validation for Step 3 (Export only - no deployment)
print("Running pre-flight validation for export...")
print()

# Check 1: Source workspace URL configured
if not source_workspace_url:
    raise ValueError("source_workspace_url not configured")
print(f"✅ Source workspace: {source_workspace_url}")

# Check 2: Volume paths exist
try:
    dbutils.fs.ls(exported_path)
    print(f"✅ Export path accessible: {exported_path}")
except:
    print(f"⚠️  Export path will be created: {exported_path}")

try:
    dbutils.fs.ls(transformed_path)
    print(f"✅ Transform path accessible: {transformed_path}")
except:
    print(f"⚠️  Transform path will be created: {transformed_path}")

# Check 3: Inventory exists
try:
    dbutils.fs.ls(inventory_path)
    print(f"✅ Inventory path accessible: {inventory_path}")
except Exception as e:
    raise ValueError(f"Inventory path not found: {inventory_path}. Run Step 1 first.")

print()
print("✅ Pre-flight validation passed - proceeding with export & transform")


## Cell 3: Discover Dashboards

In [ ]:
print("\n" + "="*80)
print("STEP 1: LOAD APPROVED INVENTORY")
print("="*80)

# Load approved inventory CSV (created in Step 2)
print(f"\n📂 Loading approved inventory from: {inventory_path}\n")

# DEBUG: Show which inventory is being loaded
print(f"🔍 DEBUG - Inventory Configuration:")
print(f"   inventory_path_rel (from widget): {dbutils.widgets.get('inventory_path')}")
print(f"   Full inventory_path: {inventory_path}")
print(f"   Expected: Should be 'dashboard_inventory_approved' (not 'dashboard_inventory')")

import pandas as pd

# Read the approved inventory CSV
# Note: Unity Catalog volumes should NOT use /dbfs prefix in serverless compute
csv_path = f"{inventory_path}/inventory_approved.csv"
inventory_df = pd.read_csv(csv_path)
dashboard_ids = inventory_df['dashboard_id'].tolist()

print(f"\n✅ Loaded {len(dashboard_ids)} approved dashboard(s)\n")
print(f"   CSV location: {csv_path}")
print(f"   Dashboards to export: {len(dashboard_ids)}")

# DEBUG: Show ALL dashboard IDs from inventory
print(f"\n🔍 DEBUG - ALL {len(dashboard_ids)} dashboard IDs from inventory:")
for i, dash_id in enumerate(dashboard_ids):
    print(f"   {i+1}. {dash_id}")

# DEBUG: Check for duplicates
if len(dashboard_ids) != len(set(dashboard_ids)):
    print(f"\n⚠️  WARNING: Found duplicate dashboard IDs in inventory!")
    duplicates = [d for d in dashboard_ids if dashboard_ids.count(d) > 1]
    print(f"   Duplicates: {set(duplicates)}")

if not dashboard_ids:
    print("\n⚠️  No dashboards in approved inventory!")
    print("   Please run Step 2 to review and approve dashboards.")
    raise ValueError("Approved inventory is empty")

## Cell 4: Export Dashboards & Permissions

In [ ]:
print("\n" + "="*80)
print("STEP 2: EXPORTING DASHBOARDS & PERMISSIONS")
print("="*80)

# Connect to source workspace
print("\n🔗 Connecting to source workspace...")
source_client = create_workspace_client('source')
print(f"   ✅ Connected\n")

if not dashboard_ids:
    print("\n⚠️  No dashboards to export")
    raise ValueError("No dashboards in approved inventory")

export_results = []

files_created = []  # Track files created

for i, dashboard_id in enumerate(dashboard_ids, 1):
    print(f"\n[{i}/{len(dashboard_ids)}] Exporting: {dashboard_id}")
    
    try:
        # Export dashboard JSON
        json_content, display_name, clean_name = export_dashboard(source_client, dashboard_id)
        
        # DEBUG: Show filename that will be created
        json_file = f"{exported_path}/dashboard_{dashboard_id}_{clean_name}.lvdash.json"
        print(f"   🔍 DEBUG - Will create file: dashboard_{dashboard_id}_{clean_name}.lvdash.json")
        
        # Save JSON
        write_volume_file(json_file, json_content)
        print(f"   ✅ Exported JSON: {display_name}")
        files_created.append(f"dashboard_{dashboard_id}_{clean_name}.lvdash.json")
        
        # Get and save permissions if enabled
        if capture_permissions:
            permissions = get_dashboard_permissions(source_client, dashboard_id)
            permissions['display_name'] = display_name
            
            perm_file = f"{exported_path}/dashboard_{dashboard_id}_{clean_name}_permissions.json"
            write_volume_file(perm_file, json.dumps(permissions, indent=2))
            
            acl_count = len(permissions.get('access_control_list', []))
            print(f"   🔐 Permissions: {acl_count} ACL(s)")
        
        # Get and save schedules if enabled
        if capture_schedules:
            schedules = get_dashboard_schedules(source_client, dashboard_id)
            schedules['display_name'] = display_name
            
            sched_file = f"{exported_path}/dashboard_{dashboard_id}_{clean_name}_schedules.json"
            write_volume_file(sched_file, json.dumps(schedules, indent=2))
            
            sched_count = len(schedules.get('schedules', []))
            subs_count = sum(len(s.get('subscriptions', [])) for s in schedules.get('schedules', []))
            print(f"   📅 Schedules: {sched_count}, Subscriptions: {subs_count}")
        
        export_results.append({
            'dashboard_id': dashboard_id,
            'name': display_name,
            'status': 'success',
            'json_file': json_file
        })
        
    except Exception as e:
        print(f"   ❌ Error: {e}")
        export_results.append({
            'dashboard_id': dashboard_id,
            'name': dashboard_id,
            'status': 'failed',
            'error': str(e)
        })

successful = len([r for r in export_results if r['status'] == 'success'])
print(f"\n✅ Successfully exported: {successful}/{len(dashboard_ids)}")

# DEBUG: Show all files that were created
print(f"\n🔍 DEBUG - Files created during export ({len(files_created)}):")
for i, fname in enumerate(files_created, 1):
    print(f"   {i}. {fname}")

# DEBUG: Check if any dashboard created multiple files
print(f"\n🔍 DEBUG - Checking for duplicate dashboard_ids in filenames:")
dashboard_id_count = {}
for fname in files_created:
    # Extract dashboard_id from filename (format: dashboard_{id}_{name})
    parts = fname.split('_', 2)
    if len(parts) >= 2:
        dash_id = parts[1]
        dashboard_id_count[dash_id] = dashboard_id_count.get(dash_id, 0) + 1

duplicates_found = {k: v for k, v in dashboard_id_count.items() if v > 1}
if duplicates_found:
    print(f"   ⚠️  FOUND DUPLICATES: Some dashboard IDs created multiple files!")
    for dash_id, count in duplicates_found.items():
        print(f"      Dashboard ID {dash_id}: {count} files")
        matching_files = [f for f in files_created if dash_id in f]
        for f in matching_files:
            print(f"         - {f}")
else:
    print(f"   ✅ No duplicates - each dashboard created exactly 1 file")

## Cell 4.5: Generate Consolidated CSV Files (Performance Optimization)

**Why CSV?** Reading 1 CSV file is 60x faster than reading 50+ individual JSON files.

This cell consolidates all permissions and schedules into single CSV files for faster loading in Step 4.

In [ ]:
print("\n" + "="*80)
print("GENERATING CONSOLIDATED CSV FILES (Performance Optimization)")
print("="*80)

import pandas as pd
import json

# Get capture flags (with fallback if not defined from earlier cells)
try:
    _capture_permissions = capture_permissions
except NameError:
    _capture_permissions = dbutils.widgets.get('capture_permissions').lower() == 'true'
    print(f"   ⚠️  capture_permissions loaded from widget: {_capture_permissions}")

try:
    _capture_schedules = capture_schedules
except NameError:
    _capture_schedules = dbutils.widgets.get('capture_schedules').lower() == 'true'
    print(f"   ⚠️  capture_schedules loaded from widget: {_capture_schedules}")

# Get exported_path (with fallback if not defined from earlier cells)
try:
    _exported_path = exported_path
except NameError:
    volume_base = dbutils.widgets.get('volume_base')
    exported_path_rel = dbutils.widgets.get('exported_path')
    _exported_path = f"{volume_base}/{exported_path_rel}"
    print(f"   ⚠️  exported_path loaded from widget: {_exported_path}")

# Only generate CSVs if we captured permissions or schedules
if not _capture_permissions and not _capture_schedules:
    print("\n⚠️  Permissions and schedules capture disabled - skipping CSV generation")
else:
    # ========================================================================
    # CONSOLIDATE PERMISSIONS INTO CSV
    # ========================================================================
    if _capture_permissions:
        print("\n📋 Consolidating permissions into CSV...")
        
        try:
            # Find all permission JSON files
            all_files = dbutils.fs.ls(_exported_path)
            perm_files = [f for f in all_files if '_permissions.json' in f.name]
            
            print(f"   Found {len(perm_files)} permission files")
            
            # Build rows for CSV
            permissions_rows = []
            for idx, perm_file in enumerate(perm_files, 1):
                print(f"   [{idx}/{len(perm_files)}] Processing {perm_file.name[:60]}...", end='', flush=True)
                try:
                    # Read JSON
                    content = dbutils.fs.head(perm_file.path, 10485760)
                    perm_data = json.loads(content)
                    
                    dashboard_id = perm_data.get('dashboard_id', 'unknown')
                    dashboard_name = perm_data.get('display_name', dashboard_id)
                    
                    # Extract each permission from access_control_list
                    for acl in perm_data.get('access_control_list', []):
                        # Get the principal (user, group, or service principal)
                        principal = acl.get('user_name') or acl.get('group_name') or acl.get('service_principal_name', '')
                        
                        # Get principal type
                        if acl.get('user_name'):
                            principal_type = 'user'
                        elif acl.get('group_name'):
                            principal_type = 'group'
                        elif acl.get('service_principal_name'):
                            principal_type = 'service_principal'
                        else:
                            principal_type = 'unknown'
                        
                        # FIX: all_permissions is a list of STRINGS, not dicts!
                        # e.g. ["CAN_READ", "CAN_MANAGE"] not [{"permission_level": "CAN_READ"}]
                        for perm_level in acl.get('all_permissions', []):
                            permissions_rows.append({
                                'dashboard_id': dashboard_id,
                                'dashboard_name': dashboard_name,
                                'principal': principal,
                                'principal_type': principal_type,
                                'permission_level': perm_level  # perm_level IS the string
                            })
                    print(" ✅")
                except Exception as e:
                    print(f" ❌ Error: {str(e)[:50]}")
            
            # Create DataFrame and save CSV
            if permissions_rows:
                df_permissions = pd.DataFrame(permissions_rows)
                csv_content = df_permissions.to_csv(index=False)
                
                permissions_csv_path = f"{_exported_path}/all_permissions.csv"
                
                # Write using helper function (works with UC volumes)
                from helpers.volume_utils import write_volume_file
                write_volume_file(permissions_csv_path, csv_content)
                
                print(f"\n   ✅ Created CSV with {len(permissions_rows)} permission entries")
                print(f"   📄 File: {permissions_csv_path}")
            else:
                print(f"\n   ⚠️  No permissions data found")
                
        except Exception as e:
            print(f"\n   ❌ Error creating permissions CSV: {e}")
            import traceback
            traceback.print_exc()
    
    # ========================================================================
    # CONSOLIDATE SCHEDULES INTO CSV
    # ========================================================================
    if _capture_schedules:
        print("\n📅 Consolidating schedules into CSV...")
        
        try:
            # Find all schedule JSON files
            all_files = dbutils.fs.ls(_exported_path)
            sched_files = [f for f in all_files if '_schedules.json' in f.name]
            
            print(f"   Found {len(sched_files)} schedule files")
            
            # Build rows for CSV
            schedules_rows = []
            for idx, sched_file in enumerate(sched_files, 1):
                print(f"   [{idx}/{len(sched_files)}] Processing {sched_file.name[:60]}...", end='', flush=True)
                try:
                    # Read JSON
                    content = dbutils.fs.head(sched_file.path, 10485760)
                    sched_data = json.loads(content)
                    
                    dashboard_id = sched_data.get('dashboard_id', 'unknown')
                    
                    # Extract each schedule
                    for schedule in sched_data.get('schedules', []):
                        schedule_id = schedule.get('schedule_id', '')
                        cron = schedule.get('cron_schedule', {}).get('cron_expression', '')
                        timezone = schedule.get('cron_schedule', {}).get('timezone_id', '')
                        paused = schedule.get('pause_status', '') == 'PAUSED'
                        
                        # Get subscriptions count
                        subscriptions = schedule.get('subscriptions', [])
                        
                        schedules_rows.append({
                            'dashboard_id': dashboard_id,
                            'schedule_id': schedule_id,
                            'cron_expression': cron,
                            'timezone': timezone,
                            'paused': paused,
                            'subscriptions_count': len(subscriptions),
                            'subscriptions_json': json.dumps(subscriptions)
                        })
                    print(" ✅")
                except Exception as e:
                    print(f" ❌ Error: {str(e)[:50]}")
            
            # Create DataFrame and save CSV
            if schedules_rows:
                df_schedules = pd.DataFrame(schedules_rows)
                csv_content = df_schedules.to_csv(index=False)
                
                schedules_csv_path = f"{_exported_path}/all_schedules.csv"
                
                # Write using helper function (works with UC volumes)
                from helpers.volume_utils import write_volume_file
                write_volume_file(schedules_csv_path, csv_content)
                
                print(f"\n   ✅ Created CSV with {len(schedules_rows)} schedule entries")
                print(f"   📄 File: {schedules_csv_path}")
            else:
                print(f"\n   ⚠️  No schedules data found")
                
        except Exception as e:
            print(f"\n   ❌ Error creating schedules CSV: {e}")
            import traceback
            traceback.print_exc()

print("\n" + "="*80)
print("CSV GENERATION COMPLETE")
print("="*80)

## Cell 5: Transform Dashboards

In [ ]:
# ============================================================================
# ARCHIVE EXISTING FILES (before writing new ones)
# ============================================================================
print("\n" + "="*80)
print("ARCHIVING EXISTING FILES")
print("="*80)

if archive_old_files:
    from helpers import archive_old_files as archive_helper
    
    # Archive ALL existing transformed files (clean slate before writing new ones)
    print("\n📦 Archiving existing transformed files...")
    try:
        transform_archive = archive_helper(
            source_folder=transformed_path,
            file_pattern="*.lvdash.json",
            archive_subfolder="archive",
            min_age_minutes=0  # Archive ALL files regardless of age
        )
        
        if 'error' in transform_archive:
            print(f"   ⚠️  Warning: {transform_archive['error']}")
        else:
            print(f"   ✅ Archived {transform_archive['archived_count']} file(s)")
            if transform_archive['archived_count'] > 0:
                print(f"   📁 Archive: {transform_archive['archive_path']}")
    except Exception as e:
        print(f"   ⚠️ Archive failed: {e}")
        print("   Continuing with processing...")
    
    # Archive permission files
    print("\n📦 Archiving existing permission files...")
    try:
        perm_archive = archive_helper(
            source_folder=exported_path,
            file_pattern="*_permissions.json",
            archive_subfolder="archive",
            min_age_minutes=0
        )
        print(f"   ✅ Archived {perm_archive.get('archived_count', 0)} file(s)")
    except Exception as e:
        print(f"   ⚠️ Archive failed: {e}")
    
    # Archive schedule files
    print("\n📦 Archiving existing schedule files...")
    try:
        sched_archive = archive_helper(
            source_folder=exported_path,
            file_pattern="*_schedules.json",
            archive_subfolder="archive",
            min_age_minutes=0
        )
        print(f"   ✅ Archived {sched_archive.get('archived_count', 0)} file(s)")
    except Exception as e:
        print(f"   ⚠️ Archive failed: {e}")
    
    print("\n✅ Archive complete - folder cleared for new files")
else:
    print("\n⚠️  Archiving disabled (--var=\"archive_old_files=true\" to enable)")

# ============================================================================
# TRANSFORM DASHBOARDS
# ============================================================================
print("\n" + "="*80)
print("TRANSFORMING DASHBOARDS")
print("="*80)

if not config['transformation']['enabled']:
    print("\n⚠️  Transformation disabled in config")
elif not export_results or successful == 0:
    print("\n⚠️  No dashboards to transform")
else:
    # Load mapping CSV
    mapping_csv_path = get_path('volume_base') + "/" + config['transformation']['mapping_csv']
    print(f"\n📋 Loading mappings: {mapping_csv_path}")
    
    try:
        mappings = load_mapping_csv(mapping_csv_path)
        print(f"   ✅ Loaded {len(mappings)} mapping rule(s)\n")
        
    except Exception as e:
        print(f"   ❌ Failed to load CSV: {e}")
        raise
    
    # Transform each dashboard
    transform_results = []
    successful_exports = [r for r in export_results if r['status'] == 'success']
    
    for i, result in enumerate(successful_exports, 1):
        name = result['name']
        json_file = result['json_file']
        
        print(f"[{i}/{len(successful_exports)}] Transforming: {name}")
        
        try:
            # Read exported JSON
            from helpers.volume_utils import read_volume_file
            json_content = read_volume_file(json_file)
            
            # Apply transformations
            transformed = transform_dashboard_json(json_content, mappings)
            
            # Save transformed version
            filename = Path(json_file).name
            transformed_file = f"{transformed_path}/{filename}"
            write_volume_file(transformed_file, transformed)
            
            print(f"   ✅ Transformed")
            
            transform_results.append({
                'dashboard_id': result['dashboard_id'],
                'name': name,
                'status': 'success'
            })
            
        except Exception as e:
            print(f"   ❌ Error: {e}")
            transform_results.append({
                'dashboard_id': result['dashboard_id'],
                'name': name,
                'status': 'failed'
            })
    
    successful_transforms = len([r for r in transform_results if r['status'] == 'success'])
    
    print("\n" + "="*80)
    print("SUMMARY")
    print("="*80)
    print(f"\n✅ Exported: {successful}/{len(dashboard_ids)}")
    print(f"✅ Transformed: {successful_transforms}/{successful}")
    
    # DEBUG: Final file count verification
    print(f"\n🔍 DEBUG - Final File Count Verification:")
    print(f"   Inventory dashboards: {len(dashboard_ids)}")
    print(f"   Files created by export: {len(files_created)}")
    print(f"   Files transformed: {successful_transforms}")
    
    # List actual files in transformed folder
    try:
        actual_files = dbutils.fs.ls(transformed_path)
        json_files = [f for f in actual_files if f.name.endswith('.json') and not f.path.endswith('/')]
        print(f"   ACTUAL files in transformed folder: {len(json_files)}")
        
        if len(json_files) != len(dashboard_ids):
            print(f"\n⚠️  MISMATCH DETECTED!")
            print(f"      Expected: {len(dashboard_ids)} files")
            print(f"      Actual: {len(json_files)} files")
            print(f"      Difference: {len(json_files) - len(dashboard_ids)} extra files")
    except Exception as e:
        print(f"   Could not list transformed folder: {e}")
    
    print(f"\n📁 Files ready at: {transformed_path}")
    print(f"\n▶️  Next: Run Step 4 - Bundle_04_Generate_and_Deploy.ipynb")